# Agentic patterns & context engineering

**Week 10 · Session 2

Yesterday you built the loop by hand (notebook 01, steps 1–6). Today, two things:

**Part A — the five patterns.** The same task, climbed rung by rung up the autonomy
ladder, so you can see what each rung actually costs.

**Part B — context engineering.** What is in the window at step 7, why it rots, and
the two fixes.

This notebook is self-contained — run it fresh, nothing from yesterday is needed.

> Tool descriptions are covered in the **measuring_agents** notebook (Step 6), where we
> can prove the difference with numbers instead of asserting it.

In [ ]:
%pip install -q openai tavily-python python-dotenv

In [1]:
import json, os, time, getpass, copy
from pathlib import Path
from openai import OpenAI

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass

for var, prompt in [("OPENAI_API_KEY", "OpenAI API key: "),
                    ("TAVILY_API_KEY", "Tavily API key: ")]:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(prompt)

client = OpenAI()
MODEL = "gpt-4.1-mini"
PRICE_IN, PRICE_OUT = 0.40, 1.60          # $ per 1M tokens, for the cost column


def cost_of(usage):
    return usage.prompt_tokens / 1e6 * PRICE_IN + usage.completion_tokens / 1e6 * PRICE_OUT


print("ready")

ready


---
# Part A — The five patterns

One question, climbed rung by rung:

> *"A customer wrote in: 'my checkout keeps failing and I've been charged twice.'
> What should we do?"*

Watch what each rung can and cannot do with it.

In [2]:
TICKET = ("A customer wrote in: 'my checkout keeps failing and I've been charged "
          "twice.' What should we do?")

## Pattern 1 — Basic responder

One call in, one answer out. No tools, no loop, no state.

In [3]:
def pattern1_basic(question):
    t0 = time.time()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
    )
    return {"answer": r.choices[0].message.content,
            "llm_calls": 1, "cost": cost_of(r.usage),
            "seconds": round(time.time() - t0, 1)}


p1 = pattern1_basic(TICKET)
print(p1["answer"][:700])
print(f"\n-> {p1['llm_calls']} call · {p1['seconds']}s · ${p1['cost']:.5f}")

Here's a suggested response and action plan:

1. Apologize and acknowledge the issue:
"Thank you for reaching out and letting us know. I’m sorry to hear that your checkout is failing and that you were charged twice."

2. Verify the charges and transaction:
- Check your payment system for the customer’s account or email to confirm if there are indeed duplicate charges.
- Identify the order(s) associated with these payments.

3. Resolve the issue:
- If duplicate charges were made, initiate a refund promptly for the extra charge.
- Assist the customer with completing their order successfully.
- Investigate and fix any checkout issues on the website or app.

4. Provide clear next steps and conta

-> 1 call · 4.1s · $0.00037


Generic advice. It has never seen your systems, your past incidents, or this
customer's account — so it produces the *shape* of a good answer with none of the
substance.

**Diagnostic:** if you replaced the LLM with a very fast human typist who knows nothing
about your company, would the program work the same way? Yes → you are at pattern 1.

---
## Pattern 2 — Router

The LLM makes exactly **one** decision — which branch — and your code does everything
else. Hugely underrated: deterministic where it matters, flexible exactly where you
want it.

In [4]:
ROUTES = {
    "billing":   "Escalate to payments-oncall. Pull the charge log before replying.",
    "technical": "Create a bug ticket, attach the session trace, reply with an ETA.",
    "account":   "Send the self-serve password/account recovery flow.",
    "other":     "Route to the general support queue for a human to triage.",
}


def pattern2_router(question):
    """The LLM only classifies. Everything after that is ordinary code."""
    t0 = time.time()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content":
                   f"Classify this support ticket into exactly one of: "
                   f"{', '.join(ROUTES)}.\n\nTICKET: {question}\n\n"
                   f'Return JSON: {{"route": "...", "urgency": "high|normal|low", '
                   f'"why": "one sentence"}}'}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    decision = json.loads(r.choices[0].message.content)
    # --- from here on, zero AI. Your code, your rules, fully testable.
    action = ROUTES.get(decision["route"], ROUTES["other"])
    return {"answer": f"[{decision['route']}/{decision['urgency']}] {action}",
            "decision": decision, "llm_calls": 1, "cost": cost_of(r.usage),
            "seconds": round(time.time() - t0, 1)}


p2 = pattern2_router(TICKET)
print(json.dumps(p2["decision"], indent=2))
print("\naction taken:", p2["answer"])
print(f"\n-> {p2['llm_calls']} call · {p2['seconds']}s · ${p2['cost']:.5f}")

{
  "route": "billing",
  "urgency": "high",
  "why": "The customer is experiencing payment issues and double charges, which directly affect billing and require urgent resolution."
}

action taken: [billing/high] Escalate to payments-oncall. Pull the charge log before replying.

-> 1 call · 1.5s · $0.00009


In [5]:
# The whole point of a router: the branches are code, so they are testable.
for t in ["I forgot my password",
          "The app crashes when I upload a PDF",
          "I was charged twice for one order",
          "Do you ship to Norway?"]:
    d = pattern2_router(t)["decision"]
    print(f"{d['route']:<10} {d['urgency']:<7} {t}")

account    high    I forgot my password
technical  high    The app crashes when I upload a PDF
billing    high    I was charged twice for one order
other      low     Do you ship to Norway?


Four tickets, four correct routes, and **every downstream action is deterministic**.
If `billing` is wrong, you fix a prompt. If the billing *handler* is wrong, you fix
ordinary Python and write a unit test for it.

> **If you are building your first LLM feature and it has to be reliable, build a
> router, not an agent.** You can always add autonomy later.

This is the Week 13 support-ticket project, in twenty lines.

---
## Pattern 3 — Tool calling

Now the model picks *which* tools, *what arguments*, and *when to stop*. This is
yesterday's loop, condensed.

In [6]:
from tavily import TavilyClient
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# --- a tiny fake internal system, so the agent has something real to reach for
ACCOUNTS = {
    "C-8817": {"name": "Dana Whitfield", "plan": "pro",
               "charges": [{"id": "ch_01", "amount": 49.00, "status": "succeeded"},
                           {"id": "ch_02", "amount": 49.00, "status": "succeeded"}],
               "note": "two identical charges 40s apart on the same cart"},
}


def lookup_account(customer_id: str) -> str:
    """Look up a customer's plan and recent charges."""
    a = ACCOUNTS.get(customer_id.strip().upper())
    if not a:
        return f"No account '{customer_id}'. Known ids: {', '.join(ACCOUNTS)}"
    charges = "; ".join(f"{c['id']} ${c['amount']} {c['status']}" for c in a["charges"])
    return f"{a['name']} · plan={a['plan']} · charges: {charges} · note: {a['note']}"


def search_web(query: str) -> str:
    """Search the live web."""
    res = tavily.search(query=query, max_results=4)
    return "\n".join(f"- {r['title']}\n  {r['url']}\n  {r['content'][:250]}"
                     for r in res["results"]) or "No results."


TOOLS = [
    {"type": "function", "function": {
        "name": "lookup_account",
        "description": ("Look up a customer's plan and recent charges by account id "
                        "(format C-NNNN). Use this before making any claim about "
                        "billing."),
        "parameters": {"type": "object",
                       "properties": {"customer_id": {"type": "string"}},
                       "required": ["customer_id"]}}},
    {"type": "function", "function": {
        "name": "search_web",
        "description": ("Search the live web for current information. Use ONLY for "
                        "facts after your training cutoff. One focused query per call."),
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
]
DISPATCH = {"lookup_account": lookup_account, "search_web": search_web}


def pattern3_tools(question, system="You are a support engineer. Investigate before "
                                   "you advise. Cite what you looked up.",
                   tools=TOOLS, dispatch=DISPATCH, max_steps=6, verbose=True):
    t0, calls, cost = time.time(), 0, 0.0
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        cost += cost_of(r.usage)
        m = r.choices[0].message
        messages.append(m)
        if not m.tool_calls:
            return {"answer": m.content, "llm_calls": step, "tool_calls": calls,
                    "cost": cost, "seconds": round(time.time() - t0, 1)}
        for c in m.tool_calls:
            calls += 1
            args = json.loads(c.function.arguments or "{}")
            if verbose:
                print(f"  [{step}] {c.function.name}({args})")
            try:
                out = dispatch[c.function.name](**args)
            except Exception as e:
                out = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": c.id,
                             "content": str(out)[:3000]})
    return {"answer": "(step budget exhausted)", "llm_calls": max_steps,
            "tool_calls": calls, "cost": cost, "seconds": round(time.time() - t0, 1)}


p3 = pattern3_tools(TICKET + " The account id is C-8817.")
print("\n" + p3["answer"][:800])
print(f"\n-> {p3['llm_calls']} calls · {p3['tool_calls']} tools · "
      f"{p3['seconds']}s · ${p3['cost']:.5f}")

  [1] lookup_account({'customer_id': 'C-8817'})

The customer on account C-8817 is on the Pro plan. I see two identical charges of $49 that succeeded, about 40 seconds apart, on the same cart.

It appears the customer's double charge issue is confirmed. Here's what we should do:

1. Apologize for the inconvenience and acknowledge the duplicate charges.
2. Offer to refund one of the charges immediately.
3. Investigate why the checkout is failing despite the charges going through successfully.
4. Advise the customer to try again after a short wait or provide alternative steps to complete their purchase.

Would you like me to proceed with issuing a refund and drafting a message to the customer?

-> 2 calls · 1 tools · 2.9s · $0.00039


Now it actually *looked*. It found two identical charges and can say something true
about this specific customer.

That is the jump from pattern 2 to 3: the model stopped choosing a label and started
choosing **actions**.

---
## Pattern 4 — Multi-agent

Three specialists in sequence: an **investigator**, a **writer**, and a **critic** that
can send the work back.

Everyone wants to start here. Watch the cost column before you do.

In [8]:
def agent_step(role_system, task, tools=None, dispatch=None, max_steps=4):
    """One specialist doing one job. Returns (text, cost)."""
    if tools:
        r = pattern3_tools(task, system=role_system, tools=tools,
                           dispatch=dispatch, max_steps=max_steps, verbose=False)
        return r["answer"], r["cost"]
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": role_system},
                  {"role": "user", "content": task}])
    return resp.choices[0].message.content, cost_of(resp.usage)


INVESTIGATOR = ("You are a support investigator. Use your tools to establish the FACTS "
                "only. Output a terse bulleted list of what you verified. No advice.")
WRITER = ("You are a support agent. Using ONLY the facts given, write a reply to the "
          "customer: acknowledge, explain, state the concrete next step.")
CRITIC = ("You review draft support replies. Check: does every claim trace to a stated "
          "fact? Is the next step concrete? Is it under 120 words?\n"
          'Return JSON: {"verdict": "approve"|"revise", "problems": ["..."]}')


def pattern4_multi(question, max_revisions=2):
    t0, total = time.time(), 0.0

    facts, c = agent_step(INVESTIGATOR, question, TOOLS, DISPATCH); total += c
    print("INVESTIGATOR:\n", facts[:400], "\n")

    draft, c = agent_step(WRITER, f"FACTS:\n{facts}\n\nTICKET: {question}"); total += c

    for attempt in range(1, max_revisions + 1):
        review_raw, c = agent_step(CRITIC + "\nReturn ONLY JSON.",
                                   f"FACTS:\n{facts}\n\nDRAFT:\n{draft}"); total += c
        try:
            review = json.loads(review_raw[review_raw.find("{"):review_raw.rfind("}") + 1])
        except Exception:
            review = {"verdict": "approve", "problems": []}
        wc = len(draft.split())
        print(f"CRITIC (round {attempt}): {review['verdict']}  "
              f"[draft is {wc} words]  {review.get('problems')}")
        if review["verdict"] == "approve":
            break
        draft, c = agent_step(
            WRITER, f"FACTS:\n{facts}\n\nYour draft:\n{draft}\n\n"
                    f"Fix these problems: {review['problems']}"); total += c

    return {"answer": draft, "cost": total, "seconds": round(time.time() - t0, 1)}


p4 = pattern4_multi(TICKET + " The account id is C-8817.")
print("\nFINAL REPLY:\n" + p4["answer"])
print(f"\n-> {p4['seconds']}s · ${p4['cost']:.5f}")

INVESTIGATOR:
 - Customer on a Pro plan.
- Two identical charges of $49.0 each succeeded.
- Charges occurred about 40 seconds apart on the same cart.
- Checkout attempts resulted in two successful charges. 

CRITIC (round 1): approve  [draft is 96 words]  []

FINAL REPLY:
Hello,

Thank you for reaching out. I see that your account on the Pro plan was charged twice, with two successful charges of $49.00 each occurring about 40 seconds apart during the same checkout attempt. This likely caused the duplicate charges you experienced.

To resolve this, the next step is to review these transactions in detail and issue a refund for the duplicate charge. I will escalate this to our billing team right away and keep you updated once the refund is processed.

Please let me know if you have any other questions in the meantime.

-> 5.3s · $0.00058


Look at the word count the critic just printed against the "under 70 words" rule it
was given. **If it approved a 110-word draft, the critic was wrong** — and that is the
most useful thing on this slide: a critic is an LLM too. It misses things, and it costs
you a full extra call every time it does.

(Re-run the cell if it approved first time; on a tighter rubric you will see a revision
round.)

Better output when it works — but look at the cost and the clock against pattern 3.

**What multi-agent buys you:** specialisation, and an independent reviewer.
**What it costs:** N× the tokens, N× the latency, and a much harder debugging story —
when the final answer is wrong, which of the three was at fault?

Use it when one agent's context genuinely cannot hold the job, or when you want a
critic. Not because it sounds more advanced.

---
## Pattern 5 — Autonomous

The model writes code it was never given, runs it, reads the failure, and revises.
Loops until the tests pass.

> ⚠️ **We execute model-generated code below.** This is a restricted namespace, not a
> real sandbox — fine for a classroom, never for production. In production: a
> container, no network, no credentials, a timeout.

In [9]:
TASK = ("Write a Python function `refund_amount(charges)` that takes a list of dicts "
        "with keys 'id', 'amount', 'status' and returns the total to refund: the sum "
        "of DUPLICATE succeeded charges (same amount appearing more than once), "
        "counting each duplicate beyond the first. Return a float.")

TESTS = """
assert refund_amount([]) == 0
assert refund_amount([{'id':'a','amount':49.0,'status':'succeeded'}]) == 0
assert refund_amount([{'id':'a','amount':49.0,'status':'succeeded'},
                      {'id':'b','amount':49.0,'status':'succeeded'}]) == 49.0
assert refund_amount([{'id':'a','amount':49.0,'status':'succeeded'},
                      {'id':'b','amount':49.0,'status':'succeeded'},
                      {'id':'c','amount':49.0,'status':'succeeded'}]) == 98.0
assert refund_amount([{'id':'a','amount':49.0,'status':'succeeded'},
                      {'id':'b','amount':49.0,'status':'failed'}]) == 0
"""


def run_code(source, tests):
    """Execute generated code + assertions. Returns (passed, message)."""
    ns = {}
    try:
        exec(source, ns)          # restricted namespace -- NOT a sandbox
        exec(tests, ns)
        return True, "all assertions passed"
    except AssertionError as e:
        return False, f"AssertionError: {e or 'a test failed'}"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


def pattern5_autonomous(task, tests, max_rounds=4):
    t0, total = time.time(), 0.0
    messages = [
        {"role": "system", "content":
         "You write Python. Return ONLY the function source, no markdown fences, "
         "no explanation, no tests."},
        {"role": "user", "content": task},
    ]
    for rnd in range(1, max_rounds + 1):
        r = client.chat.completions.create(model=MODEL, messages=messages)
        total += cost_of(r.usage)
        src = r.choices[0].message.content.strip()
        src = src.replace("```python", "").replace("```", "").strip()
        messages.append({"role": "assistant", "content": src})

        ok, msg = run_code(src, tests)
        print(f"  round {rnd}: {'PASS' if ok else 'FAIL'} — {msg}")
        if ok:
            return {"source": src, "rounds": rnd, "cost": total,
                    "seconds": round(time.time() - t0, 1)}
        # the failure IS the next prompt -- this is the whole pattern
        messages.append({"role": "user", "content":
                         f"Your code failed:\n{msg}\n\nFix it. Return ONLY the "
                         f"corrected function source."})
    return {"source": src, "rounds": max_rounds, "cost": total,
            "seconds": round(time.time() - t0, 1), "failed": True}


p5 = pattern5_autonomous(TASK, TESTS)
print("\n" + p5["source"])
print(f"\n-> {p5['rounds']} round(s) · {p5['seconds']}s · ${p5['cost']:.5f}")

  round 1: PASS — all assertions passed

def refund_amount(charges):
    from collections import defaultdict
    amount_counts = defaultdict(int)
    for charge in charges:
        if charge['status'] == 'succeeded':
            amount_counts[charge['amount']] += 1
    refund = 0.0
    for amount, count in amount_counts.items():
        if count > 1:
            refund += amount * (count - 1)
    return float(refund)

-> 1 round(s) · 2.0s · $0.00018


It may well pass on the first round — the task is not that hard, and then you have
seen a code generator, not a self-repairing agent.

**Run the next cell to see the actual pattern.** It seeds a deliberately broken first
attempt, so the repair loop always fires.

In [10]:
def pattern5_with_seeded_bug(task, tests, max_rounds=4):
    """Same loop, but we plant a broken first attempt so the repair edge always fires."""
    t0, total = time.time(), 0.0

    # A plausible-looking bug: counts every succeeded charge, not just the duplicates.
    broken = ("def refund_amount(charges):\n"
              "    return sum(c['amount'] for c in charges if c['status'] == 'succeeded')")
    print("  seeded attempt:")
    for line in broken.split("\n"):
        print("     ", line)

    messages = [
        {"role": "system", "content":
         "You write Python. Return ONLY the function source, no markdown fences, "
         "no explanation, no tests."},
        {"role": "user", "content": task},
        {"role": "assistant", "content": broken},
    ]
    src = broken
    for rnd in range(1, max_rounds + 1):
        ok, msg = run_code(src, tests)
        print(f"  round {rnd}: {'PASS' if ok else 'FAIL'} — {msg}")
        if ok:
            return {"source": src, "rounds": rnd, "cost": total,
                    "seconds": round(time.time() - t0, 1)}

        # the failure IS the next prompt -- this is the whole pattern
        messages.append({"role": "user", "content":
                         f"Your code failed:\n{msg}\n\nFix it. Return ONLY the "
                         f"corrected function source."})
        r = client.chat.completions.create(model=MODEL, messages=messages)
        total += cost_of(r.usage)
        src = r.choices[0].message.content.strip()
        src = src.replace("```python", "").replace("```", "").strip()
        messages.append({"role": "assistant", "content": src})

    return {"source": src, "rounds": max_rounds, "cost": total,
            "seconds": round(time.time() - t0, 1), "failed": True}


p5b = pattern5_with_seeded_bug(TASK, TESTS)
print("\nrepaired:\n" + p5b["source"])
print(f"\n-> repaired in {p5b['rounds']} round(s) · ${p5b['cost']:.5f}")

  seeded attempt:
      def refund_amount(charges):
          return sum(c['amount'] for c in charges if c['status'] == 'succeeded')
  round 1: FAIL — AssertionError: 
  round 2: PASS — all assertions passed

repaired:
def refund_amount(charges):
    from collections import Counter
    succeeded = [c['amount'] for c in charges if c['status'] == 'succeeded']
    counts = Counter(succeeded)
    return float(sum(amount * (count - 1) for amount, count in counts.items() if count > 1))

-> repaired in 2 round(s) · $0.00017


There it is. No fixed tool set: the agent wrote the code it needed, ran it, **read its
own error message, and revised**. That feedback edge — failure becomes the next prompt
— is the entire pattern.

This is Week 13's code-generation & self-repair project, in forty lines.

---
## The ladder, with numbers

In [11]:
rows = [
    ("1  Basic responder", p1["llm_calls"], "-", p1["seconds"], p1["cost"],
     "generic advice, no facts"),
    ("2  Router",          p2["llm_calls"], "-", p2["seconds"], p2["cost"],
     "correct branch, code does the rest"),
    ("3  Tool calling",    p3["llm_calls"], p3["tool_calls"], p3["seconds"], p3["cost"],
     "looked up the real account"),
    ("4  Multi-agent",     "~6",            "~2", p4["seconds"], p4["cost"],
     "reviewed, revised, customer-ready"),
    ("5  Autonomous",      p5["rounds"],    "-", p5["seconds"], p5["cost"],
     "wrote and repaired its own code"),
]
print(f"{'pattern':<20}{'LLM':>5}{'tools':>7}{'secs':>7}{'cost $':>10}   outcome")
print("-" * 88)
base = p1["cost"]
for name, llm, tools, secs, cost, outcome in rows:
    print(f"{name:<20}{str(llm):>5}{str(tools):>7}{secs:>7.1f}{cost:>10.5f}   {outcome}")
print("-" * 88)
print(f"pattern 5 costs {p5['cost']/base:.0f}x pattern 1, "
      f"pattern 4 costs {p4['cost']/base:.0f}x")

pattern               LLM  tools   secs    cost $   outcome
----------------------------------------------------------------------------------------
1  Basic responder      1      -    4.1   0.00037   generic advice, no facts
2  Router               1      -    1.5   0.00009   correct branch, code does the rest
3  Tool calling         2      1    2.9   0.00039   looked up the real account
4  Multi-agent         ~6     ~2    5.3   0.00058   reviewed, revised, customer-ready
5  Autonomous           1      -    2.0   0.00018   wrote and repaired its own code
----------------------------------------------------------------------------------------
pattern 5 costs 0x pattern 1, pattern 4 costs 2x


**Pick the lowest rung that solves your problem.** Autonomy you don't need is risk —
and cost, and latency — you didn't have to take.

For this ticket, rung 3 was enough. Rung 4 produced a nicer reply for several times the
price; whether that trade is worth it depends on whether a human was going to edit the
draft anyway.

---
# Part B — Context engineering

For a single call you write a prompt. For an agent you curate a **window that changes
on every step** — and most of what ends up in it was written by a tool, not by you.

## B1 — Anatomy of a system prompt

Six blocks. Let's prove they matter by running the same task with and without them.

In [ ]:
LAZY = "You are a helpful assistant."

STRUCTURED = """You are a support engineer for an e-commerce platform.

TOOL POLICY
- Always call lookup_account before making any claim about a customer's billing.
- Use search_web only for facts after your training cutoff. One query per call.
- Never state a charge amount you have not looked up.

CONSTRAINTS
- Never promise a refund timeline. Say "our payments team will confirm the timing".

STOPPING CONDITION
- Stop as soon as you have verified the facts and can state one next step.
  Do not keep gathering context.

OUTPUT CONTRACT
Return exactly:
DIAGNOSIS: <one sentence>
EVIDENCE: <what you looked up>
NEXT STEP: <one concrete action>

FAILURE BEHAVIOUR
- If a tool fails twice, stop and report what you tried. Never invent an account
  detail."""

print("LAZY prompt:", len(LAZY), "chars |  STRUCTURED:", len(STRUCTURED), "chars")

In [ ]:
q = TICKET + " The account id is C-8817."

print("########## LAZY")
lazy = pattern3_tools(q, system=LAZY)
print("\n" + lazy["answer"][:600])
print(f"\n-> {lazy['llm_calls']} LLM calls · {lazy['tool_calls']} tool calls · "
      f"${lazy['cost']:.5f}")

print("\n\n########## STRUCTURED")
structured = pattern3_tools(q, system=STRUCTURED)
print("\n" + structured["answer"][:600])
print(f"\n-> {structured['llm_calls']} LLM calls · {structured['tool_calls']} tool "
      f"calls · ${structured['cost']:.5f}")

Compare three things:

1. **Did it look things up, or assert?** The tool policy forces evidence.
2. **Is the output parseable?** `DIAGNOSIS / EVIDENCE / NEXT STEP` can be consumed by
   downstream code. Prose cannot.
3. **Did it stop?** The stopping condition is the block nobody writes and everybody
   needs — an agent without one runs until it hits your step cap, which *looks* like a
   bug in the loop and is actually a gap in the prompt.

The two blocks people most often miss are **STOPPING CONDITION** and **FAILURE
BEHAVIOUR**. Without the second, a failed tool call gets papered over with a plausible
invention, because that is what a model does when it has nothing.

In [ ]:
# Drop ONE block and watch the behaviour change. This is the cell to run live.
no_stop = STRUCTURED.replace(
    """STOPPING CONDITION
- Stop as soon as you have verified the facts and can state one next step.
  Do not keep gathering context.

""", "")

# An open-ended request is what exposes a missing stopping condition. A narrow
# one resolves in a single lookup either way and proves nothing.
OPEN_ENDED = ("Investigate account C-8817's double charge. Research the common "
              "industry causes of duplicate card charges, and any known issues with "
              "major payment providers, so we understand this fully.")

a = pattern3_tools(OPEN_ENDED, system=no_stop, max_steps=8, verbose=False)
b = pattern3_tools(OPEN_ENDED, system=STRUCTURED, max_steps=8, verbose=False)

print(f"{'':<28}{'tool calls':>12}{'cost $':>10}")
print("-" * 50)
print(f"{'without stopping condition':<28}{a['tool_calls']:>12}{a['cost']:>10.5f}")
print(f"{'with stopping condition':<28}{b['tool_calls']:>12}{b['cost']:>10.5f}")
print("-" * 50)
if a["tool_calls"] > b["tool_calls"]:
    print(f"The unbounded version did {a['tool_calls'] - b['tool_calls']} more tool "
          f"call(s) and cost {a['cost']/b['cost']:.1f}x as much.")
else:
    print("Same this run -- agents are non-deterministic. Re-run it; the unbounded\n"
          "version is the one that sprawls, and it sprawls more the longer the task.")

---
## B2 — Watch the window grow

Here is the thing nobody shows you: **the whole conversation is re-sent on every
turn.** Step 6 pays for steps 1–5 all over again.

Let's instrument the loop and look.

In [ ]:
def run_instrumented(question, system=STRUCTURED, max_steps=8):
    """Same loop, but records what the window looks like at every step."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    steps, cum_cost = [], 0.0

    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        cum_cost += cost_of(r.usage)
        m = r.choices[0].message
        messages.append(m)

        # how much of the window is raw tool output right now?
        tool_chars = sum(len(x["content"]) for x in messages
                         if isinstance(x, dict) and x.get("role") == "tool")
        other_chars = (len(system) + len(question))
        steps.append({
            "step": step,
            "sent": r.usage.prompt_tokens,          # <- what you PAY for this turn
            "cum_cost": cum_cost,
            "msgs": len(messages),
            "tool_chars": tool_chars,
            "other_chars": other_chars,
        })
        if not m.tool_calls:
            break
        for c in m.tool_calls:
            args = json.loads(c.function.arguments or "{}")
            try:
                out = DISPATCH[c.function.name](**args)
            except Exception as e:
                out = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": c.id,
                             "content": str(out)[:3000]})
    return steps, messages, m.content


# A question that forces several searches, so the window really fills.
BIG = ("Research the three most common causes of duplicate card charges in online "
       "checkout systems, then check account C-8817 against them.")
steps, msgs, answer = run_instrumented(BIG)

print(f"{'step':>5}{'tokens sent':>13}{'msgs':>6}{'cumulative $':>14}   window")
print("-" * 78)
peak = max(s["sent"] for s in steps)
for s in steps:
    bar = "#" * int(38 * s["sent"] / peak)
    print(f"{s['step']:>5}{s['sent']:>13}{s['msgs']:>6}{s['cum_cost']:>14.5f}   {bar}")
print("-" * 78)
print(f"first turn: {steps[0]['sent']} tokens   last turn: {steps[-1]['sent']} tokens"
      f"   ({steps[-1]['sent']/steps[0]['sent']:.1f}x)")
print(f"total prompt tokens paid for: {sum(s['sent'] for s in steps)}")

In [ ]:
# What is actually IN there at the end?
last = steps[-1]
tool_chars, other = last["tool_chars"], last["other_chars"]
total = tool_chars + other
print(f"raw tool output : {tool_chars:>6} chars  ({tool_chars/total:>4.0%})")
print(f"system + question: {other:>6} chars  ({other/total:>4.0%})")
print()
print("The biggest thing in the window is the part you never wrote,")
print("and would never choose to keep -- raw tool output from earlier steps.")

Two things to take from that table:

1. **Tokens sent grows every turn**, because the entire history is re-sent. Cost is
   roughly quadratic in the number of steps, not linear. This is why a runaway loop
   gets expensive so fast.
2. **Raw tool output dominates.** You wrote the system prompt and the question; a tool
   wrote the rest, and you never proofread it.

The failure modes from here: **context rot** (quality degrades as the window fills,
long before the advertised limit), **lost in the middle** (instructions buried
mid-window get ignored), and **distraction** (irrelevant history pulls the model
off-task).

---
## B3 — Fix 1: shape the tool output

The cheapest saving available, and it needs no model at all. **Your code decides what
gets through.**

In [ ]:
def search_web_shaped(query: str) -> str:
    """Same search, but only the fields that matter, and hard-capped."""
    res = tavily.search(query=query, max_results=4)
    out = []
    for r in res["results"][:3]:                     # 3 not 4
        out.append(f"- {r['title']}\n  {r['url']}\n  {r['content'][:120]}")
    return "\n".join(out)[:700] or "No results."     # hard ceiling


probe = "common causes of duplicate card charges in checkout"
raw = search_web(probe)
shaped = search_web_shaped(probe)
print(f"raw    : {len(raw):>5} chars")
print(f"shaped : {len(shaped):>5} chars   ({1 - len(shaped)/len(raw):.0%} smaller)")
print("\n--- shaped ---")
print(shaped[:400])

No model involved, no tokens spent deciding. You just chose not to send 70% of it.

**Do not hand a model a raw API response.** Return the four fields that matter.

---
## B4 — Fix 2: summarise and drop

For long runs, replace old raw tool output with a two-line summary once you are past
it. The agent keeps the *conclusion* and loses the *transcript*.

In [ ]:
def compact(messages, keep_last=2, max_chars=200):
    """Replace old raw tool results with short summaries. Keeps the most recent intact.

    Real systems use an LLM to summarise. Truncation is the cheap version, and it
    already recovers most of the saving.
    """
    tool_idx = [i for i, m in enumerate(messages)
                if isinstance(m, dict) and m.get("role") == "tool"]
    stale = set(tool_idx[:-keep_last]) if len(tool_idx) > keep_last else set()
    out = []
    for i, m in enumerate(messages):
        if i in stale:
            short = m["content"][:max_chars]
            out.append({**m, "content": f"[summarised] {short}..."})
        else:
            out.append(m)
    return out


def chars_of(ms):
    return sum(len(m["content"]) if isinstance(m, dict) and isinstance(m.get("content"), str)
               else 0 for m in ms)


# Demonstrate the mechanism on a fixed conversation, so the saving is visible
# regardless of how many steps your live run happened to take.
demo = [{"role": "system", "content": STRUCTURED},
        {"role": "user", "content": BIG}]
for i in range(5):
    demo.append({"role": "tool", "tool_call_id": f"c{i}",
                 "content": search_web(f"duplicate charge cause {i}")})

b4, af = chars_of(demo), chars_of(compact(demo, keep_last=2))
n_tool = sum(1 for m in demo if m.get("role") == "tool")
print(f"conversation with {n_tool} tool results")
print(f"  before compaction : {b4:>6} chars")
print(f"  after  compaction : {af:>6} chars   ({1 - af/b4:.0%} smaller)")
print(f"  ({n_tool - 2} stale results summarised, 2 most recent kept intact)")

# ...and on your actual run above
rb, ra = chars_of(msgs), chars_of(compact(msgs, keep_last=1))
n_real = sum(1 for m in msgs if isinstance(m, dict) and m.get("role") == "tool")
print(f"\nyour live run had {n_real} tool result(s): "
      f"{rb} -> {ra} chars ({1 - ra/rb:.0%} smaller)")
if n_real <= 1:
    print("  (too few to compact -- compaction pays off on LONG runs, which is "
          "exactly when you need it)")
print("\nSame conclusions available to the model. Far less transcript.")

In [ ]:
# End to end: the same run, with both fixes applied.
DISPATCH_SHAPED = {**DISPATCH, "search_web": search_web_shaped}


def run_lean(question, system=STRUCTURED, max_steps=8):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    total_sent, cum_cost = 0, 0.0
    for step in range(1, max_steps + 1):
        messages = compact(messages)                      # fix 2
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        total_sent += r.usage.prompt_tokens
        cum_cost += cost_of(r.usage)
        m = r.choices[0].message
        messages.append(m)
        if not m.tool_calls:
            return {"answer": m.content, "sent": total_sent, "cost": cum_cost,
                    "steps": step}
        for c in m.tool_calls:
            args = json.loads(c.function.arguments or "{}")
            try:
                out = DISPATCH_SHAPED[c.function.name](**args)   # fix 1
            except Exception as e:
                out = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": c.id,
                             "content": str(out)[:3000]})
    return {"answer": "(budget)", "sent": total_sent, "cost": cum_cost,
            "steps": max_steps}


lean = run_lean(BIG)
naive_sent = sum(s["sent"] for s in steps)
naive_cost = steps[-1]["cum_cost"]

print(f"{'':<12}{'prompt tokens':>16}{'cost $':>10}{'steps':>7}")
print("-" * 46)
print(f"{'naive':<12}{naive_sent:>16}{naive_cost:>10.5f}{len(steps):>7}")
print(f"{'lean':<12}{lean['sent']:>16}{lean['cost']:>10.5f}{lean['steps']:>7}")
print("-" * 46)
if naive_sent:
    print(f"tokens saved: {1 - lean['sent']/naive_sent:>.0%}")
print("\n--- lean answer ---")
print(lean["answer"][:600])

Same question, same model, same tools. The difference is entirely **what you chose to
put in the window**.

> Numbers vary run to run — agents are non-deterministic. Run it twice and look at the
> direction, not the decimal.

---
## What you take forward

**Part A — the ladder**

| Rung | Use it when |
|---|---|
| 1 Basic | pure language work, no facts needed |
| 2 Router | you need reliability and the branches are knowable |
| 3 Tool calling | the model needs to look things up or act |
| 4 Multi-agent | one context can't hold it, or you want a critic |
| 5 Autonomous | the method genuinely can't be specified in advance |

**Part B — context engineering**

```
Six prompt blocks   role · tool policy · constraints ·
                    stopping condition · output contract · failure behaviour
The window grows    the whole history is re-sent every turn
Fix 1               shape tool output in YOUR code, before the model sees it
Fix 2               summarise and drop old observations
```

**Next:** `measuring_agents.ipynb` — now that you can build these, how do you know any
of it works?

---
## Exercises

**1. Add a fifth route** to the router and three test tickets for it. Notice you can
unit-test the handler — that is the whole appeal of rung 2.

**2. Remove the critic** from pattern 4 and compare the draft quality and the cost. Was
the critic worth it?

**3. Give pattern 5 a harder spec** — partial refunds, or currency rounding — and watch
how many repair rounds it needs. Cap `max_rounds` and see it give up.

**4. Delete a different prompt block.** Drop `OUTPUT CONTRACT` and try to parse the
result with code. Drop `FAILURE BEHAVIOUR` and make a tool raise — does it invent?

**5. Real compaction.** Replace the truncation in `compact()` with an actual LLM
summary call. Does the answer quality hold up? What did the summary cost you?